## Preprocessing AQI dataset
   -  This Jupyter Notebook contains python code pipeline used to clean, transfer and integrate AQI dataset chunks downloaded from cpcb.gov.in.

(source: https://airquality.cpcb.gov.in/ccr/#/caaqm-dashboard-all/caaqm-landing/caaqm-data-repository)


###    -Filtering data 

In [1]:
import pandas as pd
import numpy as np

#### As it appears in 'Raw Datasets/AQI_data_raw' in the repo, the AQI data is in chunks. Data is from GPCB weather center, Maninagar, Ahmedabad.

In [248]:
df1= pd.read_csv('/Users/meetdave/Public/AqiPred/AQI_raw/Raw_data_1Hr_2025_site_308_Maninagar_Ahmedabad_GPCB_1Hr.csv')

In [258]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 8017 entries, 0 to 8016
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    8017 non-null   str    
 1   PM2.5   8017 non-null   float64
dtypes: float64(1), str(1)
memory usage: 275.1 KB


In [259]:
df1.head()


,Time,PM2.5
0,2025-01-01 00:00:00,68.85
1,2025-01-01 01:00:00,58.16
2,2025-01-01 02:00:00,45.12
3,2025-01-01 03:00:00,40.15
4,2025-01-01 04:00:00,36.47


In [63]:
df1.columns


Index(['Timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)'], dtype='str')

#### Removing irrelavant columns:

In [251]:
df1.drop(columns= [ 'PM10 (µg/m³)' ,'NO (µg/m³)',
       'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)',
       'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)',
       'O Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)',
       'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)',
       'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)'], inplace = True)

In [253]:
df1.reset_index(inplace=True, drop=True)

In [254]:
#beacuse pm2.5 has complex name, rename all columns

df1.columns = ['Time', 'PM2.5']

#### The process of interpolation: 
#### linear interpolation fills NaN values by drawing a straight line between the two nearest data points and assumes the change between points is constant. 




In [ ]:
df1['PM2.5'] = df1['PM2.5'].interpolate(method='linear', limit=3)

#### Now in the column 'PM2.5' only those values are left which have the gap oo more than 3. Droping rows by these gaps.

In [256]:
df1 = df1.dropna(subset=['PM2.5'])

In [257]:
df1.reset_index(inplace=True, drop = True)

In [260]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 8017 entries, 0 to 8016
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    8017 non-null   str    
 1   PM2.5   8017 non-null   float64
dtypes: float64(1), str(1)
memory usage: 275.1 KB


In [261]:
df1.head()

,Time,PM2.5
0,2025-01-01 00:00:00,68.85
1,2025-01-01 01:00:00,58.16
2,2025-01-01 02:00:00,45.12
3,2025-01-01 03:00:00,40.15
4,2025-01-01 04:00:00,36.47


In [262]:
df1.to_pickle('2.5_25.pkl')

#### As seen in previous cells, data files are loaded, cleaned, filtered, and saved in a pickle file. 

#### Pickle files (.pkl) can handle big datasets. These files take entire dataframe object, convert them into stream of bytes, which keeps datatypes and makes it faster than CSVs.

## Integrating data chunks
### -Datatype correction

#### From here each file is loaded, cleaned, concatenated with previous bit and saved into the same previous file. 

#### loading 1st data chunk

In [410]:

tmp = pd.read_pickle("/Users/meetdave/Public/AqiPred/AQI_processed/2015.pkl")

In [420]:
tmp.info()
tmp.head()

<class 'pandas.DataFrame'>
DatetimeIndex: 81046 entries, 2015-01-28 17:00:00 to 2025-12-31 23:00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pm2.5   81046 non-null  float32
dtypes: float32(1)
memory usage: 949.8 KB


,pm2.5
valid_time,
2015-01-28 17:00:00,36.599998
2015-01-28 18:00:00,41.139999
2015-01-28 19:00:00,58.110001
2015-01-28 20:00:00,82.239998
2015-01-28 21:00:00,106.690002


#### If not the first iteration, reseting the index of aggregated data

In [413]:

tmp.reset_index(inplace=True, drop=True)

#### Concatenating current processed dataframe df to previously saved .pkl file containing one/ more than one dataframes combined.

In [412]:

tmp = pd.concat([tmp, df])

#### Now saving the file

In [415]:

tmp.to_pickle('/Users/meetdave/Public/AqiPred/AQI_processed/2015.pkl')

#### here aggregation of chunks ends

In [417]:
tmp.columns = ['valid_time', 'pm2.5']

In [418]:
tmp.info()

<class 'pandas.DataFrame'>
RangeIndex: 81046 entries, 0 to 81045
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype        
---  ------      --------------  -----        
 0   valid_time  81046 non-null  datetime64[s]
 1   pm2.5       81046 non-null  float32      
dtypes: datetime64[s](1), float32(1)
memory usage: 949.9 KB


In [419]:
#rounding off the frequency to hourly intervals 

tmp = tmp.set_index('valid_time')

In [424]:
tmp.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 95767 entries, 2015-01-28 17:00:00 to 2025-12-31 23:00:00
Freq: h
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pm2.5   81046 non-null  float32
dtypes: float32(1)
memory usage: 1.1 MB


In [440]:
tmp.head(9)

,pm2.5
valid_time,
2015-01-28 17:00:00,36.599998
2015-01-28 18:00:00,41.139999
2015-01-28 19:00:00,58.110001
2015-01-28 20:00:00,82.239998
2015-01-28 21:00:00,106.690002
2015-01-28 22:00:00,117.620003
2015-01-28 23:00:00,120.949997
2015-01-29 00:00:00,136.589996
2015-01-29 01:00:00,119.510002


#### Resampling from 1h gap to 6h gap because weather data is in 6h gaps

In [441]:

tmp = tmp.resample('6h').max()

In [442]:
tmp.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 15962 entries, 2015-01-28 12:00:00 to 2025-12-31 18:00:00
Freq: 6h
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pm2.5   13732 non-null  float32
dtypes: float32(1)
memory usage: 187.1 KB


In [443]:
#checkpoint (finalaqi.pkl)

In [444]:
tmp.head(10)

,pm2.5
valid_time,
2015-01-28 12:00:00,36.599998
2015-01-28 18:00:00,120.949997
2015-01-29 00:00:00,136.589996
2015-01-29 06:00:00,90.620003
2015-01-29 12:00:00,68.620003
2015-01-29 18:00:00,111.279999
2015-01-30 00:00:00,87.459999
2015-01-30 06:00:00,87.510002
2015-01-30 12:00:00,58.340000


#### checking if any value has imaginary distribution

In [446]:

tmp.describe()

,pm2.5
count,13732.000000
mean,78.914612
std,75.540604
min,0.520000
25%,39.830002
50%,58.809998
75%,91.925003
max,999.989990


In [ ]:
tmp.to_pickle('/Users/meetdave/Public/AqiPred/AQI_processed/finalaqi.pkl')

#### Till here the final aqi dataset is cleaned and integrated into a single pickle file. 